# Notebook 03 — Feature Engineering and Targets

**FIN 4600 · Lab 3 · Financial Data Analytics**

Duran, *Financial Services Technology* (3rd ed.), **Chapter 6 — Data Analytics**

---

This is the bridge between descriptive and predictive analytics, and it is
where most of the real work — and most of the real mistakes — live.

A machine learning model needs a rectangular table: one row per observation,
one column per **feature**, plus a **target** column holding the thing you are
trying to predict. Building that table from a time series is not hard. Building
it *without accidentally leaking the future into it* is hard, and it is the
difference between a model that works and a backtest that lies to you.

**What you will learn**

1. Building per-instrument features with `groupby().transform()`
2. Joining market-wide data (VIX) onto per-instrument data
3. The lag discipline — what you knew, and when you knew it
4. Look-ahead bias, demonstrated rather than described
5. Defining a target without leaking it into the features
6. Why a random train/test split is wrong for time series
7. Scaling, and why it is fitted on training data only

#### How to use this notebook

Every code cell is complete and every line carries a comment. Run the cells top to bottom
(**Run All** is fine), read the explanation above each cell, then read the comments and the
output together. Two exercises have worked solutions. When you have finished, take the Canvas
comprehension quiz for this notebook.

#### Background: features, targets and supervised learning

**Supervised learning** means learning a mapping from inputs to a known answer, using
historical examples where the answer has already been observed.

- A **feature** (also: predictor, input, explanatory variable, $X$) is something you know
  at the moment you make a decision: a stock's trailing return, its recent volatility, the
  level of the VIX.
- A **target** (also: label, outcome, response, $y$) is the thing you want to predict and
  only find out later: next month's return, or whether next month's return is positive.
- An **observation** is one row: one stock at one point in time.

The model learns $\hat{y} = f(X)$ from rows where $y$ is known (the training set), and is
judged on rows it has never seen (the test set). If $y$ is a number (a return) the task is
**regression**; if $y$ is a category (up / down, default / no default) it is
**classification**. This notebook builds both kinds of target.

The algorithm in Notebooks 04 and 05 is the easy part. The hard part, and the part that
decides whether the result is real, is this notebook: building $X$ and $y$ so that every
feature on a row was genuinely knowable *before* the target on that row happened. When that
fails, information from the future has **leaked** into the inputs, and the backtest lies.

## 0. Setup

The first cell imports the libraries, locates the repository folder so the data files can be
found wherever Jupyter was started, and sets display options. It prints nothing; it just has
to run without an error.

#### Background: the libraries (recap from Notebooks 01 and 02)

- **pandas** is the standard Python library for tables (`DataFrame`) and labelled columns
  (`Series`). Banks, asset managers and fintechs use it because it handles dates, joins,
  group-by and rolling calculations in a few readable lines.
- **NumPy** is the numerical array library underneath pandas; here it supplies `np.log` and
  `np.sqrt`.
- **matplotlib** draws charts; `pyplot` is its plotting interface.
- **pathlib.Path** builds file paths that work on Windows, macOS and Linux alike.

In [ ]:
# Path represents a file-system path; the "/" operator joins folder and file names safely on any OS
from pathlib import Path

# NumPy: fast numerical arrays and maths functions (we use np.log and np.sqrt below)
import numpy as np
# pandas: DataFrames (tables) and Series (columns) — the workhorse for this whole lab
import pandas as pd
# matplotlib's pyplot interface, used for the train/test timeline chart in section 6
import matplotlib.pyplot as plt


# Define a helper that finds the repo folder, so the notebook works however Jupyter was launched
def find_repo_root(start=None):
    # The docstring below documents the function (shown by help(find_repo_root))
    """Return the repository root — the folder that contains data/sp500_prices.csv."""
    # Start from the given folder, or the current working directory; resolve() makes it absolute
    here = Path(start or Path.cwd()).resolve()
    # Walk upwards: this folder first, then its parent, grandparent, and so on
    for candidate in [here, *here.parents]:
        # The repo root is the first folder that contains data/sp500_prices.csv
        if (candidate / "data" / "sp500_prices.csv").exists():
            # Found it — hand the folder back to the caller
            return candidate
    # If no folder matched, stop with a clear message telling the student how to fix it
    raise FileNotFoundError(
        "Could not find the repository root. In VS Code use File > Open Folder "  # message, part 1
        "and open the mtu4600-lab3-analytics folder itself, then re-run."  # part 2 (adjacent strings join)
    )


# Locate the repository root once and keep it for the rest of the notebook
REPO = find_repo_root()
# The folder holding the raw CSV files
DATA = REPO / "data"
# Apply the course's chart style file (fonts, colours, grid) to every plot
plt.style.use(REPO / "fin4600.mplstyle")

# Show up to 40 columns when a DataFrame is displayed, instead of hiding the middle ones with "..."
pd.set_option("display.max_columns", 40)
# Allow 150 characters per printed line before pandas wraps a wide table
pd.set_option("display.width", 150)

### Load the three raw files

This cell reads daily prices for 30 large S&P 500 companies, a table of company information
(including sector), and the daily VIX. `parse_dates=["date"]` turns the date text into real
`datetime64` values so that sorting, month-grouping and merging on dates work correctly.

The price table is then sorted by ticker and, within ticker, by date. **That sort matters
more here than anywhere else in the lab**: every `shift` and `rolling` call below works on row
order, not on dates. If the rows were out of order, "yesterday" would be some random other day.

Look for: 37,770 rows and 30 tickers, and the first rows all belonging to AAPL in date order.

#### Background: panel data

This dataset is a **panel** (also called longitudinal or cross-sectional time-series data):
many entities (30 tickers) each observed over many dates. It is stored in **long format**,
one row per (ticker, date) pair, with the ticker as an ordinary column. Panels have two axes
of variation, and leakage can travel along either:

- the **time-series** axis (one stock over time), where the future must never leak into the past;
- the **cross-sectional** axis (all stocks on one date), where rows from the same day are
  correlated with each other because they share the same market move.

Every feature in this notebook must therefore be computed *within* a ticker and *up to*
the current date.

In [ ]:
# Daily open/high/low/close/volume for 30 tickers; parse_dates makes "date" a datetime64 column
prices = pd.read_csv(DATA / "sp500_prices.csv", parse_dates=["date"])
# One row per company: ticker, name, sector, etc. (no dates, so no date parsing needed)
companies = pd.read_csv(DATA / "sp500_companies.csv")
# One row per trading day with the VIX "fear index" level; again parse the date column
vix = pd.read_csv(DATA / "vix_daily.csv", parse_dates=["date"])

# Sort by ticker, then date, and renumber the index 0..n-1 (drop=True discards the old index).
# shift() and rolling() rely on this row order, so it must be correct before any feature is built.
prices = prices.sort_values(["ticker", "date"]).reset_index(drop=True)
# Report the size of the panel: total rows (with thousands separator) and distinct tickers
print(f"{len(prices):,} rows, {prices['ticker'].nunique()} tickers")
# Display the first three rows to confirm the columns and the sort order
prices.head(3)

## 1. Per-instrument features with `groupby().transform()`

Every feature here must be computed **within** a ticker. If you compute a
21-day return on the whole sorted frame, the first row of MSFT will be
compared with the last row of MMM, which is nonsense.

`groupby("ticker").transform(...)` applies a function inside each group and
returns a result aligned to the original rows. It is the single most useful
pattern for panel data.

#### Background: `groupby().transform()` versus `groupby().agg()`

`groupby("ticker")` splits the table into 30 sub-tables, one per ticker. What happens next
decides the shape of the answer:

- `.agg()` / `.mean()` **collapse** each group to one row (30 rows out): useful for summaries.
- `.transform(f)` runs `f` on each group's Series and returns a result **the same length as
  the input, aligned to the original row index** (37,770 rows out). That is exactly what a
  feature column needs, so it can be assigned straight back with `prices["new"] = ...`.

Because each ticker is processed on its own, any `shift` or `rolling` window inside `f`
restarts at each ticker's first row. AAPL's last day can never bleed into AMT's first day.

#### Background: `shift()` and lagging

`s.shift(k)` moves a Series **down** by `k` rows, so row $t$ holds the value from row $t-k$;
the first `k` rows become `NaN` because there is nothing earlier to bring down.
`s.shift(-k)` moves it **up**: row $t$ then holds the value from $t+k$, a value from the
*future*. That second form is how targets are built later, and it must never be used for a feature.

With `shift` a $d$-day simple return is one line:

$$\text{momentum}_t = \frac{P_t}{P_{t-d}} - 1 = \texttt{s / s.shift(d) - 1}$$

#### Background: rolling windows

`s.rolling(n)` creates a moving window of the last `n` rows; follow it with a statistic such
as `.mean()`, `.std()` or `.max()`. By default the window is **trailing**: the value on row
$t$ uses rows $t-n+1$ to $t$ inclusive, which is today and the $n-1$ days before. The first
$n-1$ rows of each ticker are `NaN` because the window is not yet full (the "warm-up" period).

#### The features built in the next cell

| Feature | Formula | Finance idea |
|---|---|---|
| `momentum_1m/3m/6m/12m` | $P_t/P_{t-d}-1$ for $d$ = 21, 63, 126, 252 trading days | Momentum (winners keep winning over 3 to 12 months) and short-term reversal (1 month) |
| `volatility_21d/63d` | rolling std of daily returns $\times\sqrt{252}$ | Recent annualised risk (Notebook 02: variance grows with time, so SD scales with $\sqrt{T}$) |
| `px_vs_200d_ma` | $P_t / \text{mean}_{200}(P) - 1$ | Trend: above or below the 200-day moving average, a classic technical signal |
| `px_vs_52w_high` | $P_t / \max_{252}(P) - 1$ | How far below its 52-week high (always $\le 0$); anchoring studies find stocks near their high keep outperforming |
| `volume_zscore` | $(\ln V_t - \text{mean}_{63}) / \text{sd}_{63}$ on log volume | Is trading unusually heavy today, in standard deviations of its own recent history? Logs tame volume's huge right skew |

`pct_change()` is shorthand for `s / s.shift(1) - 1`, the daily simple return.

Look for in the output: the last three AAPL rows of February 2018 with every feature filled
in. Volatility around 0.2 to 0.27 is about 20 to 27% annualised; the `volume_zscore` of about
2.6 on 5 February 2018 is the "Volmageddon" sell-off, when volume spiked across the market.

In [ ]:
# A GroupBy object over the "close" column, one group per ticker; reused for several features below
grouped = prices.groupby("ticker")["close"]

# Trailing returns over several horizons — "momentum"
# Loop over (trading days, label) pairs: ~21 trading days per month, ~252 per year
for days, name in [(21, "1m"), (63, "3m"), (126, "6m"), (252, "12m")]:
    # For each ticker, price today / price `days` rows ago - 1. The default argument d=days freezes
    # the current loop value inside the lambda (otherwise every lambda would see the final value, 252).
    prices[f"momentum_{name}"] = grouped.transform(lambda s, d=days: s / s.shift(d) - 1)

# Trailing realised volatility, annualised
# Daily simple return within each ticker; each ticker's first row is NaN (no previous close)
daily_return = grouped.transform(lambda s: s.pct_change())
# Store it as a column so it can be grouped again below
prices["daily_return"] = daily_return
# 21-day rolling standard deviation of daily returns per ticker, annualised by sqrt(252)
prices["volatility_21d"] = (
    prices.groupby("ticker")["daily_return"].transform(lambda s: s.rolling(21).std())  # per-ticker 21-day SD
    * np.sqrt(252)  # daily SD x sqrt(trading days per year) = annual SD
)
# Same calculation over a 63-day (about one quarter) window: smoother, slower to react
prices["volatility_63d"] = (
    prices.groupby("ticker")["daily_return"].transform(lambda s: s.rolling(63).std())  # per-ticker 63-day SD
    * np.sqrt(252)  # annualise exactly as above
)

# Where is the price relative to its own trailing average and its 52-week high?
# Close divided by its trailing 200-day mean, minus 1: positive means trading above trend
prices["px_vs_200d_ma"] = grouped.transform(lambda s: s / s.rolling(200).mean() - 1)
# Close divided by the highest close of the last 252 days, minus 1: zero at a new high, negative below
prices["px_vs_52w_high"] = grouped.transform(lambda s: s / s.rolling(252).max() - 1)

# Is volume unusual right now, measured in its own standard deviations?
# Natural log of share volume (a Series aligned to prices); logs make volume roughly symmetric
log_volume = np.log(prices["volume"])
# assign() adds log volume as a temporary column "lv" without changing prices itself; then, per
# ticker, compute (today's lv - its trailing 63-day mean) / its trailing 63-day SD: a rolling z-score
prices["volume_zscore"] = prices.assign(lv=log_volume).groupby("ticker")["lv"].transform(
    lambda s: (s - s.rolling(63).mean()) / s.rolling(63).std()  # rolling z-score per ticker
)

# The list of model inputs; later cells select exactly these columns, so the list is the "X" definition
feature_columns = [
    "momentum_1m",  # 1-month trailing return
    "momentum_3m",  # 3-month trailing return
    "momentum_6m",  # 6-month trailing return
    "momentum_12m",  # 12-month trailing return
    "volatility_21d",  # 1-month annualised volatility
    "volatility_63d",  # 3-month annualised volatility
    "px_vs_200d_ma",  # distance from the 200-day moving average
    "px_vs_52w_high",  # distance below the 52-week high
    "volume_zscore",  # unusual-volume z-score
]

# Show the last three AAPL rows with the price and every feature, as a quick visual check
(
    # Boolean mask keeps AAPL rows; the column list picks date, close and the nine features
    prices.loc[prices["ticker"] == "AAPL", ["date", "close"] + feature_columns]
    # Keep only the three most recent rows
    .tail(3)
    # Replace the timestamp with a plain date (drops the 00:00:00) purely for neat display
    .assign(date=lambda d: d["date"].dt.date)
    # Round to 4 decimal places for readability
    .round(4)
)

### Sanity check: did the grouping actually work?

If the features had been computed across the whole frame instead of within
each ticker, the first row of each ticker after the first would carry a
nonsense value instead of `NaN`. Check it.

`groupby("ticker").head(1)` returns the first row of every ticker (30 rows). `.notna().sum()`
then counts, per column, how many of those 30 values are *not* missing. Every count should
be 0: on a ticker's first day there is no history, so every trailing feature must be `NaN`.
A non-zero count would prove that a window had reached back into the previous ticker's rows.
This is a cheap, mechanical test of the grouping, and the habit of writing such checks is
worth more than any single feature.

In [ ]:
# First row of each ticker: groupby + head(1) keeps the first row per group, in original order
first_rows = prices.groupby("ticker").head(1)
# Explain what the numbers below mean
print("Non-null feature values on each ticker's FIRST row (should all be 0):")
# notna() gives True/False per cell; sum() counts the Trues per column; to_string() prints every row
print(first_rows[feature_columns].notna().sum().to_string())

## 2. Joining market-wide data

The VIX is one number per day, shared by every instrument. A `merge` on
`date` broadcasts it across all 30 tickers for that day.

The next cell builds three market-wide features from the VIX table *before* the merge, while
the VIX is still one clean time series:

- `vix_close`: the level of the VIX, the market's implied 30-day volatility for the S&P 500 in
  percentage points (about 15 is calm; above 30 is stress).
- `vix_change_5d`: its 5-day percentage change, `pct_change(5)` = $V_t/V_{t-5}-1$.
  A rising VIX means fear is building.
- `vix_vs_63d_avg`: the level relative to its own trailing 63-day mean.

Note there is no `groupby` here: the VIX table has one row per day, so a plain
`shift`/`rolling` is already correct. It also starts in 2012, a year before the price data,
so its rolling windows are warmed up by the time the stocks begin.

#### Background: merges (recap from Notebook 01)

`DataFrame.merge(other, on="date", how="left")` is a SQL-style join. `how="left"` keeps every
row of the left table (`prices`) and attaches the matching VIX columns; a price row with no
matching VIX date would get `NaN`. This is a **many-to-one** join: 30 price rows per date
match one VIX row, so the VIX value is copied ("broadcast") to all 30 tickers. If the right
table had duplicate dates, rows would multiply silently. Hence the habit: **check the row
count before and after every merge**.

Look for: identical row counts and zero rows without a VIX value.

In [ ]:
# Take just the date and closing VIX level; copy() makes an independent table we can add columns to
vix_features = vix[["date", "vix_close"]].copy()
# 5-day percentage change in the VIX: VIX today / VIX five trading days ago - 1
vix_features["vix_change_5d"] = vix_features["vix_close"].pct_change(5)
# VIX relative to its trailing 63-day average, minus 1: positive means fear is above its recent norm
vix_features["vix_vs_63d_avg"] = (
    vix_features["vix_close"] / vix_features["vix_close"].rolling(63).mean() - 1  # level / 63-day mean - 1
)

# Remember the row count so the merge can be checked
before = len(prices)
# Left join on date: every price row keeps its place and gains the three VIX columns for that day
panel = prices.merge(vix_features, on="date", how="left")
# The counts must match; a bigger "after" would mean duplicate dates in the VIX table
print(f"Rows before merge: {before:,}   after: {len(panel):,}   (must match)")
# isna().sum() counts rows where the VIX was not found for that date
print(f"Rows with no VIX value: {panel['vix_close'].isna().sum()}")

# Add the three market-wide columns to the feature list (+= extends the list in place)
feature_columns += ["vix_close", "vix_change_5d", "vix_vs_63d_avg"]

### Attach the sector

A second left merge, this time on `ticker`, adds each company's sector. It is many-to-one
again (about 1,259 daily rows per ticker match one company row). Sector is kept for later
analysis, not used as a model feature here.

The `or "none"` trick: an empty Python list counts as false, so `[] or "none"` evaluates to the
string `"none"`. Look for: `Unmatched tickers: none`.

In [ ]:
# Attach sector for later analysis (not used as a model feature here)
# Left join on ticker: only the ticker and sector columns of companies are brought across
panel = panel.merge(companies[["ticker", "sector"]], on="ticker", how="left")
# List any tickers whose sector came back missing; an empty list falls through to the text "none"
print("Unmatched tickers:", panel.loc[panel["sector"].isna(), "ticker"].unique().tolist() or "none")

## 3. The lag discipline

This is the heart of the notebook.

Ask of every feature: **at the moment I would act on this, would I actually
have known it?**

Our features are built from rolling windows that end at the current row, so
the value in row $t$ uses prices up to and including the close on day $t$.
That is fine *if* the decision is made after the close on day $t$ and the
outcome starts on day $t+1$. It is not fine if you compare it with day $t$'s
own return — you would be using the closing price to predict itself.

The safe habit: **shift the features forward by one period**, so that row $t$
holds only information available strictly before $t$.

#### Background: what "information available at time $t$" means

Think of a timeline for one stock:

| Row | Close used in the feature | When you could act | Return you could earn |
|---|---|---|---|
| day $t$ (unlagged) | up to and including $P_t$ | after the close of day $t$ | from $P_t$ to $P_{t+1}$ |
| day $t$ (lagged by 1) | up to $P_{t-1}$ only | any time on day $t$ | from $P_{t-1}$ or the open of $t$ onwards |

Both are honest **as long as the target starts after the information ends**. The danger is
mixing them up: pairing a feature that contains $P_t$ with a target that also contains $P_t$
(such as day $t$'s own return $P_t/P_{t-1}-1$). Then the feature literally contains part of
the answer. Lagging every feature by one row (`groupby("ticker")[col].shift(1)`) is the
belt-and-braces fix: whatever the target, row $t$ holds only what was known by the end of $t-1$.
It also mirrors reality: data vendors deliver closing prices after the close, and a trade
placed on that information fills at the next available price, not at the close you just read.

The next cell lags every feature into a separate copy called `lagged`, then lines up five AAPL
days side by side. `.to_numpy()[200:205]` turns each column into a plain array and takes
positions 200 to 204 (row 200 is well past the 21-day warm-up). Look for: each value in the
lagged column equals the value one row *above* in the "as computed" column.

In [ ]:
# Work on a copy so the original panel keeps its unlagged features
lagged = panel.copy()
# Lag each feature by one row within each ticker, so row t shows the value as of day t-1
for column in feature_columns:
    # shift(1) inside groupby: each ticker's first row becomes NaN rather than borrowing another ticker's value
    lagged[column] = lagged.groupby("ticker")[column].shift(1)

# Build a small side-by-side table for five AAPL days (array positions 200-204)
comparison = pd.DataFrame(
    {  # dict of column name -> array of five values
        # AAPL closing prices for those five days
        "close": panel.loc[panel["ticker"] == "AAPL", "close"].to_numpy()[200:205],
        # 1-month momentum as originally computed (uses today's close)
        "momentum_1m (as computed)": panel.loc[panel["ticker"] == "AAPL", "momentum_1m"].to_numpy()[200:205],
        # The same feature after the one-day lag (uses yesterday's close)
        "momentum_1m (lagged)": lagged.loc[lagged["ticker"] == "AAPL", "momentum_1m"].to_numpy()[200:205],
    }
)
# Display the table rounded to 4 decimals
comparison.round(4)

The lagged column is the same series moved down one row. That one-row shift is
often all that separates a defensible model from a fraudulent one.

In the output above, the "as computed" value $-0.0228$ on the first row reappears in the
lagged column on the second row, $-0.0042$ moves from row two to row three, and so on.

A note for later cells: the `lagged` table is a demonstration. The monthly modelling table in
section 5 is built from the *unlagged* `panel`. That is still honest, because there the
features use month-end closes up to $P_t$ and the target starts *at* $P_t$ and runs to the next
month-end, so the information ends exactly where the outcome begins (the first row of the
timeline table above).

## 4. Look-ahead bias, demonstrated

Rather than take this on faith, build a deliberately cheating feature and see
what it does. A **centred** rolling window is the classic accident — it looks
innocuous and it uses future data.

#### Background: look-ahead bias and data leakage

**Data leakage** is any route by which information that would not be available at prediction
time reaches the model during training or evaluation. It produces models that look
excellent in testing and fail in production. The main forms in finance:

1. **Look-ahead bias**: a feature uses data dated after the decision point (centred windows,
   whole-sample statistics, financial statements used on their period-end date rather than
   their later publication date, revised macro data used as if it were the first release).
2. **Target leakage**: a feature is a function of the target itself, or of something that
   happens because of the target (e.g. a "days past due" field when predicting default).
3. **Train/test contamination**: test-set information shapes training, through a random split
   of time-series rows, or a scaler or feature selection fitted on all the data (section 7).
4. **Survivorship bias**: the sample only contains firms that survived. Our 30 tickers are
   large, well-known companies chosen with hindsight; a 2013 investor could not have known
   that list. Treat any "average return" from it with scepticism.

#### Background: why `center=True` cheats

`s.rolling(5).mean()` on day $t$ averages $P_{t-4},\dots,P_t$. `s.rolling(5, center=True).mean()`
puts the window's *middle* on day $t$, so it averages $P_{t-2},\dots,P_{t+2}$: two days of the
future. Centred windows are the right tool for smoothing a chart after the fact, and they are
the default in some other software, which is why the accident is common.

Now build the signal "how far is today's close above its 5-day average?" both ways, and
correlate it with tomorrow's return $r_{t+1} = P_{t+1}/P_t - 1$. Algebraically, the centred
average contains $P_{t+1}$, so when tomorrow's price jumps, the centred average rises and the
signal $P_t/\text{MA}^{c}_t - 1$ falls. The signal is mechanically negatively correlated with
tomorrow's return. That is not a pattern in markets, it is arithmetic.

The target here is built with `s.shift(-1) / s - 1`: a **negative** shift pulls tomorrow's
close up into today's row. Using the future is correct for a target (that is the thing being
predicted) and fatal for a feature.

`dropna(subset=[...])` keeps only rows where all three columns exist, and `Series.corr()`
computes the Pearson correlation of two aligned Series. Look for: an honest correlation close
to zero and a cheating one far from zero.

In [ ]:
# Honest: a trailing 5-day average, ending today
# Per ticker, the mean of today's close and the four closes before it
panel["ma5_trailing"] = panel.groupby("ticker")["close"].transform(
    lambda s: s.rolling(5).mean()  # trailing window: rows t-4 .. t
)

# Cheating: a centred 5-day average, which peeks two days into the future
# center=True labels each 5-row window by its middle row, so day t averages days t-2 .. t+2
panel["ma5_centred"] = panel.groupby("ticker")["close"].transform(
    lambda s: s.rolling(5, center=True).mean()  # centred window: rows t-2 .. t+2
)

# What we want to predict: tomorrow's return
# shift(-1) moves tomorrow's close up into today's row; divide by today's close and subtract 1
panel["next_day_return"] = panel.groupby("ticker")["close"].transform(
    lambda s: s.shift(-1) / s - 1  # next close / this close - 1
)

# Keep only rows where both signals and the target exist (drops warm-up and final rows per ticker)
check = panel.dropna(subset=["ma5_trailing", "ma5_centred", "next_day_return"])
# Honest signal: percentage distance of today's close above its trailing 5-day average
honest_signal = check["close"] / check["ma5_trailing"] - 1
# Cheating signal: the same distance, but from the centred average that contains future closes
cheating_signal = check["close"] / check["ma5_centred"] - 1

# Header for the two results
print("Correlation with tomorrow's return:")
# Pearson correlation of the honest signal with next-day return; +.4f forces a sign and 4 decimals
print(f"  Trailing 5-day average (honest):  {honest_signal.corr(check['next_day_return']):+.4f}")
# The same for the cheating signal
print(f"  Centred 5-day average (cheating): {cheating_signal.corr(check['next_day_return']):+.4f}")

The honest signal has essentially no predictive correlation, which is what
five decades of market efficiency research would lead you to expect. The
cheating signal has a large one — and it is entirely an artefact of
`center=True` letting tomorrow's price into today's average.

**If a financial model suddenly starts working well, your first hypothesis
should be a leak, not a discovery.** Go and look for it before you tell
anyone.

The actual numbers: about $-0.01$ for the honest signal and about $-0.63$ for the cheating one.
A correlation of $-0.63$ between a daily signal and the next day's stock return would be the
most valuable trading signal in history, and it is produced by one keyword argument.

The next cell deletes the cheating column with `drop(columns=[...])`, which returns the table
without that column. Removing a known leak from the working table, rather than just
"remembering not to use it", is good practice. (`ma5_trailing` and `next_day_return` stay in
`panel` but are not in `feature_columns`, so the model never sees them.)

In [ ]:
# Drop the cheating column so nobody uses it by accident
# drop(columns=...) returns a new DataFrame without that column; reassigning replaces panel
panel = panel.drop(columns=["ma5_centred"])

### Exercise 1 (worked solution)

Here are three candidate features. For each, decide whether it leaks the
future, and explain why in a sentence:

1. `close / close.rolling(60).max() - 1` — distance below the 60-day high
2. `close / close.mean() - 1` — distance from the average price over the
   whole sample
3. `volume / volume.rolling(20).mean()` — today's volume relative to recent
   volume

Write your answers in the markdown cell below.

This is a written exercise, so the worked answers follow in the next cell.

**Worked answers.**

1. `close / close.rolling(60).max() - 1` **does not leak.** A plain `rolling(60)` window is
   trailing: on day $t$ it covers days $t-59$ to $t$, all of which have happened by the close of
   day $t$. (Computed within each ticker, and used with a target that starts after day $t$'s close,
   it is the same kind of feature as `px_vs_52w_high`.)
2. `close / close.mean() - 1` **leaks.** `close.mean()` is one number computed over the whole
   sample, including every price up to February 2018. On a day in 2013 the feature already
   "knows" where the price will average over the next five years: a stock that rises strongly
   will look cheap relative to that mean in its early years. Any whole-sample statistic (mean,
   standard deviation, min, max, a z-score using the full-sample mean) has the same flaw. The
   honest version is an expanding or rolling mean, which only uses the past.
3. `volume / volume.rolling(20).mean()` **does not leak**, with one condition. The window is
   trailing and includes today's volume, which is only fully known at the close. It is honest if
   the decision is taken after the close and the target starts after it; if you meant to trade
   *during* day $t$, lag it by a day with `shift(1)`, exactly as in section 3.

**Discussion.** The common thread: ask where each window ends. Trailing windows end today;
centred windows end in the future; whole-sample statistics end at the last row of the dataset.

## 5. From daily to monthly, and defining the target

Daily returns are mostly noise. Monthly observations give a model a better
shot and cut the sample to a size that trains in seconds, which matters in a
lab.

We take the **last observation of each month** for the features, and define
the target as the return of the **following** month.

#### Background: resampling daily data to monthly

`dt.to_period("M")` converts each date to a monthly **Period** such as `2014-03`: a label for the
whole month, not a point in time. Grouping by `["ticker", "month"]` and taking `.last()` keeps
the final trading day of each month for each ticker. Because every feature is a *trailing*
statistic, the month-end row already summarises the month (21-day volatility on the last day of
March is March's volatility). This is the same idea as `resample("ME").last()` from Notebook 01,
done per ticker.

Two details worth knowing:

- `GroupBy.last()` returns the last **non-missing** value of each column within the group, not
  literally the last row. For our features that makes no difference (once a trailing feature has
  warmed up it stays filled), but for a column that can be `NaN` on the final day of a month it
  can quietly pull in an earlier day's value.
- `.reset_index()` turns the two group keys back into ordinary columns; the result has one row
  per ticker-month.

#### Background: dropping the partial final month

The data run from 8 February 2013 to 7 February 2018, so the first and last months are
partial. The last one matters. February 2018 holds only five trading days, so its "month-end"
row is really 7 February. January 2018's next-month target would then be the return from
31 January to 7 February: one week dressed up as a month, a fake target that would sit in the
test set beside genuine monthly ones.

The cell therefore checks whether the last date in the data is a month end.
`pd.offsets.BMonthEnd(0)` rolls a date forward to the last business day of its own month, and
leaves it alone if it is already there. If adding it moves the last date, the final month is
incomplete and is dropped. The first month (February 2013) is also partial, but harmlessly:
its month-end row is a genuine month end, and the 12-month momentum and 52-week-high features
need a year of warm-up, so no row from 2013 reaches the modelling table anyway.

Look for: the message saying February 2018 was dropped, and 1,800 rows, about 60 months per
ticker.

In [ ]:
# Make sure rows are in ticker, date order before taking "the last row of each month"
panel = panel.sort_values(["ticker", "date"])
# Monthly period label for every row, e.g. Timestamp 2014-03-31 -> Period 2014-03
panel["month"] = panel["date"].dt.to_period("M")

# Collapse daily rows to one row per ticker per month
monthly = (
    # Group on both keys, so each group is one ticker in one month
    panel.groupby(["ticker", "month"])
    # Last non-missing value of each column in the group = the month-end observation
    .last()
    # Move ticker and month out of the index back into columns
    .reset_index()
    # Sort so shift() within each ticker steps month by month in calendar order
    .sort_values(["ticker", "month"])
)

# The data stop on 7 February 2018, so February 2018 is a partial month of only five trading days.
# Last trading date in the data
last_date = panel["date"].max()
# BMonthEnd(0) rolls a date forward to its month's last business day (or leaves it if already there)
if last_date < last_date + pd.offsets.BMonthEnd(0):
    # The final month is incomplete: drop it, so no target is built from a five-day "month"
    monthly = monthly[monthly["month"] < last_date.to_period("M")]
    # Say what was dropped, so the reader knows
    print(f"Dropped the partial final month {last_date.to_period('M')} (data end {last_date.date()})")

# Size of the daily table
print(f"Daily panel: {len(panel):,} rows")
# Size of the monthly table, and the approximate months per ticker (// is integer division)
print(f"Monthly panel: {len(monthly):,} rows "
      f"({monthly['ticker'].nunique()} tickers x ~{len(monthly) // monthly['ticker'].nunique()} months)")  # second half of the message

### Define the targets

#### Background: forward-looking targets

A target is *supposed* to come from the future: it is what happens after the decision. With
month-end close $P_m$ the next-month return is

$$y_m = \frac{P_{m+1}}{P_m} - 1 = \texttt{s.shift(-1) / s - 1} \quad\text{(within each ticker)}$$

The row for month $m$ therefore pairs features known at the close of month $m$ with an outcome
that begins at that same close. Information and outcome meet at exactly one instant and never
overlap, which is the whole game.

The cell builds three targets:

- `next_month_return`: a continuous number, for **regression**.
- `next_month_up`: 1 if that return is positive, 0 otherwise, for **classification**. The
  comparison `> 0` returns True/False; `.astype("Int64")` (capital I) converts to pandas'
  *nullable* integer type, which, unlike NumPy's `int64`, can hold a missing value `<NA>`.
  This matters because `NaN > 0` evaluates to False, so without the next line every ticker's
  final month, which has no next month, would be wrongly labelled "down". The `.loc[...] = pd.NA`
  line puts the missing values back.
- `next_month_volatility`: next month's 21-day realised volatility, the month-end value one row
  ahead. A risk forecasting target, used in Notebook 05.

Each ticker's last month has no "next month", so its targets are missing and those rows will
drop out of the modelling table.

Look for: in the AAPL rows, `next_month_return` on one row is the change from that row's
`close` to the next row's `close` (63.0571 to 63.2371 is $+0.0029$), and `next_month_up` is 1
exactly when it is positive.

In [ ]:
# Target: the return of the NEXT month, computed from month-end closes
# Per ticker: next month-end close (shift(-1) pulls it up one row) / this month-end close - 1
monthly["next_month_return"] = monthly.groupby("ticker")["close"].transform(
    lambda s: s.shift(-1) / s - 1  # next close / this close - 1
)
# Binary label: True/False for "return > 0", stored as nullable integers 1/0
monthly["next_month_up"] = (monthly["next_month_return"] > 0).astype("Int64")
# Where the return is unknown (each ticker's final month), mark the label missing instead of 0
monthly.loc[monthly["next_month_return"].isna(), "next_month_up"] = pd.NA

# A second, different target: next month's realised volatility
# Per ticker, the 21-day volatility measured at the NEXT month-end (shift(-1) = one row ahead)
monthly["next_month_volatility"] = monthly.groupby("ticker")["volatility_21d"].shift(-1)

# Show the first 15 monthly rows (all AAPL) with the price, one feature and the targets
(
    # Pick the columns to display
    monthly[["ticker", "month", "close", "momentum_3m", "next_month_return", "next_month_up"]]
    # First 15 rows
    .head(15)
    # Show the Period as plain text such as "2013-02"
    .astype({"month": str})
    # Round numbers to 4 decimals
    .round(4)
)

### Check the target for leakage

The target must not be derivable from any feature. The quickest check is a
correlation scan: a feature correlating unusually strongly with the target is
either a genuine discovery or, far more likely, a leak.

#### Background: information coefficient (IC) style correlation checks

In quantitative investing, the **information coefficient** is the correlation between a
signal measured today and the return that follows. Practitioners usually compute it
*cross-sectionally*, often as a Spearman rank correlation across all stocks within each month,
then average over months. A realistic monthly IC for a good equity signal is roughly
0.02 to 0.10. Through the "fundamental law of active management"
(information ratio $\approx$ IC $\times \sqrt{\text{breadth}}$), a small IC applied to many
independent bets can still be worth money. An IC of 0.5 almost never exists.

That gives a two-sided leakage test:

- an |IC| far above about 0.1 to 0.2 for return prediction is a red flag for a leak;
- an IC near zero is normal and honest.

The next cell runs a simpler, pooled version: one Pearson correlation per feature across all
ticker-months. `dropna(subset=...)` first keeps only rows with every feature and the target
(the 12-month momentum and 252-day high need a year of warm-up, so the table starts in February
2014). `DataFrame.corrwith(series)` correlates each column with the target, giving one number
per feature. `sort_values(key=abs, ascending=False)` sorts by *absolute* size, so strong
negative correlations rank as high as strong positive ones.

Interpretation caution: the three VIX features take the same value for all 30 stocks in a
month, so their correlation is really a *market-timing* statistic based on only 47
distinct months, not 1,410 independent observations.

In [ ]:
# The modelling table: month-end rows with every feature AND the return target present; copy() to own it
model_data = monthly.dropna(subset=feature_columns + ["next_month_return"]).copy()

# Correlation of every feature with next month's return, ranked by size
correlations = (
    # The 12 feature columns
    model_data[feature_columns]
    # Pearson correlation of each column with the target Series, returned as a Series indexed by feature
    .corrwith(model_data["next_month_return"])
    # Sort by absolute value, largest first, so the sign does not affect the ranking
    .sort_values(key=abs, ascending=False)
)
# Heading
print("Correlation of each feature with next month's return:")
# Print every feature's correlation to 4 decimals
print(correlations.round(4).to_string())
# The single largest absolute correlation: the number to compare against a "too good to be true" threshold
print(f"\nLargest absolute correlation: {correlations.abs().max():.4f}")

Everything is small. That is the correct and slightly deflating result: these
features carry very little information about next month's direction. A
correlation of 0.6 here would have meant a bug, not a trading strategy.

Notebook 05 takes this honestly — and then shows a target where the same
features *do* work.

In the output above, the largest absolute correlation is about 0.099 (`vix_vs_63d_avg`, one of
the market-wide VIX features), and the momentum features are all below 0.03 in absolute value.

#### Background: base rates and class balance

The next cell reports the size and date range of the modelling table and the **class balance**
of `next_month_up`: `value_counts(normalize=True)` returns each value's *share* of rows
rather than its count. The majority-class share is the **base rate**, the accuracy of the
laziest possible model, which always predicts the most common class. A classifier is only
interesting if it beats that number, and beats it on data it has not seen.

Look for: 1,410 rows from 2014-02 to 2017-12 (47 months x 30 tickers) and the up/down split.

In [ ]:
# Number of usable ticker-months
print(f"Rows available for modelling: {len(model_data):,}")
# Earliest and latest month in the table (Periods compare in calendar order)
print(f"Date range: {model_data['month'].min()} to {model_data['month'].max()}")
# Heading; the leading \n prints a blank line first
print(f"\nClass balance of next_month_up:")
# Share of rows labelled 1 (up) and 0 (down), rounded to 3 decimals
print(model_data["next_month_up"].value_counts(normalize=True).round(3).to_string())

Roughly 58/42 in favour of "up". That base rate is the number any classifier
must beat. A model that always predicts "up" is 58% accurate and completely
useless — which is why accuracy alone is a bad metric, a point Notebook 04
develops.

## 6. Splitting the data — by time, never at random

For cross-sectional data (like the loan applications in Notebook 04) a random
split is right. For time series it is wrong, for two reasons:

1. **Temporal leakage.** If March 2015 is in training and February 2015 is in
   testing, the model has seen the future relative to its test set.
2. **Cross-sectional leakage.** Our panel has 30 tickers on the same date.
   A random split puts some of a given month's rows in training and the rest
   in testing. Since stocks are correlated within a month, the model can
   effectively read the answer off its training rows.

Split on a date. Everything before the cut trains, everything after tests.

#### Background: the time-ordering problem in train/test splits

A test set exists to imitate the future: data the model has never seen, used once to estimate
how it will perform in production. For independent observations (one loan application does
not tell you about another), a random shuffle does that well. Financial time series break the
independence assumption in two ways, the two listed above. A random split therefore scores the
model on a "future" that is partly its own past, and the reported accuracy is optimistic.

The fix used here is a single **chronological hold-out**: pick a cut-off month, train on
everything strictly before it, test on everything from it onwards. More advanced versions,
which Notebook 05 uses, are:

- **walk-forward** (expanding-window) validation, scikit-learn's `TimeSeriesSplit`: train on
  months 1 to $k$, test on $k+1$ to $k+j$, then roll forward and repeat;
- a **gap** or **embargo** between train and test when targets span several periods, so the
  last training target does not overlap the first test month.

The cell below compares `month` (a Period) with `SPLIT_DATE` using `<` and `>=`, which keeps
every month in exactly one set. `set(...).isdisjoint(...)` returns True when two sets share no
element: a direct proof that no month sits on both sides.

Look for: 840 training rows (2014-02 to 2016-05, 28 months) and 570 test rows (2016-06 to
2017-12, 19 months), roughly a 60/40 split.

In [ ]:
# The cut-off month: everything before June 2016 trains, June 2016 onwards tests
SPLIT_DATE = pd.Period("2016-06", freq="M")

# Training rows: months strictly before the cut-off (boolean mask on the month column)
train = model_data[model_data["month"] < SPLIT_DATE]
# Test rows: the cut-off month and everything after it
test = model_data[model_data["month"] >= SPLIT_DATE]

# Rows and month range of each set; >5, right-aligns the count in a 5-character field with commas
print(f"Train: {len(train):>5,} rows   {train['month'].min()} to {train['month'].max()}")
# Same summary for the test set
print(f"Test:  {len(test):>5,} rows   {test['month'].min()} to {test['month'].max()}")
# True if the set of training months and the set of test months have nothing in common
print(f"\nNo month appears in both sets: {set(train['month']).isdisjoint(set(test['month']))}")

### Picture the split

A horizontal bar chart (`ax.barh`) draws the timeline: a blue bar for the training months
followed by an orange bar for the test months, starting where the training bar ends (`left=`).
`plt.subplots` returns a figure (the canvas) and an axes object (the plot area) that every
`ax.` call draws on. `ax.text` writes the month counts in the middle of each bar.

Look for: the blue (train) block always to the left of the orange (test) block, with no
interleaving. That is what "train on the past, test on the future" looks like.

In [ ]:
# A wide, short figure with one set of axes; figsize is (width, height) in inches
fig, ax = plt.subplots(figsize=(9.5, 2.6))
# Distinct months in each set, in calendar order
train_months = sorted(train["month"].unique())
# Distinct test months, also in order
test_months = sorted(test["month"].unique())
# Blue bar at y=0, starting at 0 and as long as the number of training months
ax.barh([0], [len(train_months)], left=[0], color="#2a78d6", height=0.5, label="Train")
# Orange bar at y=0 that starts where the training bar ends
ax.barh([0], [len(test_months)], left=[len(train_months)], color="#eb6834", height=0.5, label="Test")
# Hide the y-axis ticks: there is only one bar row, so they mean nothing
ax.set_yticks([])
# Label the x-axis with its unit
ax.set_xlabel("Months since start of sample")
# Chart title stating the lesson
ax.set_title("A time-series split: train on the past, test on the future")
# Turn off the background grid for a cleaner bar
ax.grid(False)
# Legend in the lower-right corner, laid out in two columns
ax.legend(loc="lower right", ncols=2)
# Write the training month count, centred in the blue bar, in bold white text
ax.text(len(train_months) / 2, 0, f"{len(train_months)} months", ha="center", va="center",
        color="white", fontsize=9, fontweight="bold")  # text styling continues the call above
# Write the test month count, centred in the orange bar
ax.text(len(train_months) + len(test_months) / 2, 0, f"{len(test_months)} months",
        ha="center", va="center", color="white", fontsize=9, fontweight="bold")  # centring and styling
# Render the chart
plt.show()

## 7. Scaling — fit on train only

Many models need features on comparable scales. `volatility_21d` runs around
0.2 while `vix_close` runs around 15; a distance-based or regularised model
will let the VIX dominate purely because of its units.

The rule that students get wrong: **fit the scaler on the training data
only**, then apply it to the test data. Fitting on everything lets the test
set's mean and standard deviation — information from the future — influence
the training features. It is a small leak, but it is a leak, and graders
(and regulators) look for it.

#### Background: scikit-learn and `StandardScaler`

**scikit-learn** (`sklearn`) is Python's standard machine-learning library. Every tool shares
one interface: `fit` learns parameters from data, `transform` applies them, `predict` makes
predictions. Its consistency, documentation and maturity are why it is the default in industry
and in model-risk validation teams.

`StandardScaler` converts each feature to a **z-score**:

$$z = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}}$$

- `fit` computes and stores each column's mean $\mu$ and standard deviation $\sigma$
  (in `scaler.mean_` and `scaler.scale_`);
- `transform` applies the stored $\mu$ and $\sigma$ to whatever data it is given;
- `fit_transform` does both on the same data, in one step.

So the training set comes out with mean 0 and SD 1 by construction (scikit-learn uses the
population SD, dividing by $n$; pandas' `describe()` uses $n-1$, hence the printed 1.001).
The test set is scaled with the *training* $\mu$ and $\sigma$, so its means and SDs are not 0
and 1. That difference is real information: it shows how the test period differs from the
training period. The scaler returns a NumPy array without column names, so the cell wraps it
back in a DataFrame to label it.

Scaling matters for regularised linear models, logistic regression, k-nearest neighbours,
SVMs and neural networks. Tree-based models (decision trees, random forests, gradient
boosting) split on thresholds and are unaffected by it.

Look for: in the test block, `vix_close` with a mean of about $-1.01$ and SD of about $0.49$.
The 2016-2017 test period was much calmer than training (which included the August 2015 and
early 2016 sell-offs), and volatility features are also negative. A scaler fitted on all the
data would have blended that calm future into the training features.

In [ ]:
# Import the z-score scaler from scikit-learn's preprocessing module
from sklearn.preprocessing import StandardScaler

# Create an unfitted scaler (it holds no means or SDs yet)
scaler = StandardScaler()
# Learn each feature's mean and SD from TRAINING rows only, then rescale those rows; returns a NumPy array
train_scaled = scaler.fit_transform(train[feature_columns])   # fit AND transform
# Rescale the test rows with the training mean and SD; nothing is learned from the test set
test_scaled = scaler.transform(test[feature_columns])         # transform ONLY

# Heading for the training summary
print("Training features after scaling — mean ≈ 0, SD ≈ 1 by construction:")
# Wrap the array in a labelled DataFrame, run describe(), and keep only the mean and std rows
print(pd.DataFrame(train_scaled, columns=feature_columns).describe().loc[["mean", "std"]].round(3))

# Heading for the test summary, which is expected to differ
print("\nTest features after applying the SAME scaler — NOT exactly 0 and 1,")
# Second line of the heading
print("and that is correct. The test set is a different period.")
# Same summary for the test rows
print(pd.DataFrame(test_scaled, columns=feature_columns).describe().loc[["mean", "std"]].round(3))

In Notebook 04 we stop doing this by hand and use a scikit-learn `Pipeline`,
which makes the leak structurally impossible rather than merely discouraged.

### Exercise 2 (worked solution)

Add one feature of your own to `feature_columns` and rebuild `model_data`.
Some ideas:

- the spread between 21-day and 63-day volatility (a term structure of risk)
- the ratio of `momentum_1m` to `momentum_12m` (short-term reversal against
  long-term trend)
- a calendar feature such as the month number, to test the "January effect"

State in one sentence, before you compute anything, whether your feature
could leak the future. Then check its correlation with `next_month_return`.


**Worked solution.** The solution adds the first suggested feature, the **volatility term
spread** $\sigma_{21d} - \sigma_{63d}$: positive when risk over the last month is higher than
over the last quarter (risk rising), negative when it is calming down.

*Leakage statement, before computing anything:* it cannot leak, because both inputs are
trailing rolling windows that end on the month-end row, so the spread only uses prices up to
that close, and the target starts at that close.

To show a second idea cheaply, it also tests the **January effect** with a calendar dummy
equal to 1 when the *target* month is January (that is, the feature row is December). A
calendar is known in advance, so it cannot leak either.

So that later cells are unaffected, the solution works on new variables (`ex2_features`,
`ex2_data`) and leaves `feature_columns` and `model_data` exactly as they were; the file saved
in section 8 is the same one Notebooks 04 and 05 expect. It also reports the correlation on the
training months only, because a feature should be chosen without looking at the test period.


In [ ]:
# Work on a copy of the monthly table so monthly and model_data stay untouched for later cells
ex2_monthly = monthly.copy()
# Volatility term spread: 1-month minus 3-month annualised volatility (both trailing, so no leak)
ex2_monthly["vol_term_spread"] = ex2_monthly["volatility_21d"] - ex2_monthly["volatility_63d"]
# Calendar dummy: 1 if this row is December, i.e. the month being predicted is January; else 0
ex2_monthly["next_is_january"] = (ex2_monthly["month"].dt.month == 12).astype(int)

# The original 12 features plus the two new ones, as a NEW list (feature_columns is not changed)
ex2_features = feature_columns + ["vol_term_spread", "next_is_january"]
# Rebuild the modelling table exactly as in section 5, but with the extended feature list
ex2_data = ex2_monthly.dropna(subset=ex2_features + ["next_month_return"]).copy()
# Rows should match model_data, since the new features have no extra warm-up period
print(f"Rows in rebuilt table: {len(ex2_data):,} (original model_data: {len(model_data):,})")

# Loop over the two new features and report their correlation with next month's return
for ex2_name in ["vol_term_spread", "next_is_january"]:
    # Pooled correlation over all 1,410 ticker-months
    ex2_all = ex2_data[ex2_name].corr(ex2_data["next_month_return"])
    # Correlation over training months only (before SPLIT_DATE), the honest place to choose features
    ex2_train = ex2_data.loc[ex2_data["month"] < SPLIT_DATE, ex2_name].corr(
        ex2_data.loc[ex2_data["month"] < SPLIT_DATE, "next_month_return"]  # matching training targets
    )
    # Print both correlations side by side with a sign and 4 decimals
    print(f"{ex2_name:<16} corr (all months): {ex2_all:+.4f}   corr (train months only): {ex2_train:+.4f}")

# For the January dummy, compare average next-month return in January versus other months
ex2_jan = ex2_data.groupby("next_is_january")["next_month_return"].agg(["mean", "count"])
# Index 0 = February-to-December targets, 1 = January targets; mean is a decimal monthly return
print("\nAverage next-month return by target month (0 = Feb-Dec, 1 = January):")
# Round to 4 decimals and print as a small table
print(ex2_jan.round(4).to_string())

**Discussion.** Both features passed the leakage test before any number was computed, and the
results are worth reading carefully.

- **Volatility term spread.** The rebuilt table has the same 1,410 rows as `model_data`, as
  expected, since the spread needs no extra warm-up. Its correlation with next month's return
  is about $+0.002$ over all months and about $+0.035$ over the training months only: tiny
  either way. That is what an honest feature with little predictive power looks like, in line
  with the 12 original features (largest about 0.099).
- **January dummy.** This one looks far more exciting: about $-0.099$ over all months and
  about $-0.208$ over the training months, larger than any original feature. The averages show
  why: January targets averaged about $-0.6\%$ against about $+1.3\%$ for other months. But the
  120 January rows are only **4 distinct Januaries** (the target months 2015 to 2018), and the
  training set holds just two of them, January 2015 and January 2016, both bad months for US
  stocks (the stocks averaged roughly $-3.5\%$ and $-3.1\%$). Every one of the 30 tickers in
  a month shares the same market move, so 60 training rows are really 2 observations. The
  "signal" is two bad months, not a pattern, and its sign is the opposite of the classic
  January effect (which is about small stocks *rising* in January).

Two lessons: a feature can be leak-free and still misleading when the number of *independent*
events behind it is small, and a correlation that is large compared with everything else
deserves the same scepticism as a suspected leak. Count the independent events before you
believe a number.

## 8. Save the modelling table

The final code cell writes the modelling table to `data/processed/monthly_features.csv`, which
Notebooks 04 and 05 read. It saves identifiers (ticker, month, date, sector, close), the 12
features and the three targets: 20 columns in all. `mkdir(exist_ok=True)` creates the folder
only if it is missing, and `to_csv(index=False)` omits pandas' row numbers from the file. The
`month` Period is converted to text such as `2014-03` first, because CSV files have no Period
type.

Look for: 1,410 rows by 20 columns, and the list of the 12 features.

In [ ]:
# Path of the output folder, data/processed
processed = DATA / "processed"
# Create it if it does not exist yet; exist_ok=True means "no error if it already exists"
processed.mkdir(exist_ok=True)

# Columns to save, in order: identifiers, then the features, then the three targets
save_columns = (
    ["ticker", "month", "date", "sector", "close"]  # identifiers and price
    + feature_columns  # the 12 model inputs
    + ["next_month_return", "next_month_up", "next_month_volatility"]  # the targets
)
# Select those columns, turn the month Period into text, and write a CSV without the index
model_data[save_columns].assign(month=lambda d: d["month"].astype(str)).to_csv(
    processed / "monthly_features.csv", index=False  # destination file; skip the row index
)

# Confirm where the file went
print("Wrote", processed / "monthly_features.csv")
# Confirm its shape
print(f"{len(model_data):,} rows x {len(save_columns)} columns")
# List the feature names, comma-separated
print("\nFeatures:", ", ".join(feature_columns))

---

## Recap

1. `groupby().transform()` computes features within each instrument.
2. After every merge, check the row count.
3. Ask of every feature: would I have known this at the time?
4. A centred rolling window is a leak. So is any statistic computed over the
   whole sample.
5. A model that suddenly works is a leak until proven otherwise.
6. Split time series by date. Random splits leak, twice over.
7. Fit scalers on training data only.

## Your turn

Every cell in this notebook is complete. Run all of them from the top, read the explanation
above each cell and the comments inside it, and check the printed output against what the text
told you to look for. Read the worked solutions to both exercises. Then complete the **Canvas
comprehension quiz for Notebook 03**.

Commit and push your run notebook:

```
git add notebooks/03_features_and_targets.ipynb
git commit -m "Ran notebook 03"
git push
```

Next: `04_ml_credit_risk.ipynb` — supervised learning on a problem where the
signal is real and strong, so you can learn the machinery before facing the
market's noise.